In [7]:
!pip install -q pdfplumber
!pip install -q langchain
!pip install -q langchain-community
!pip install -q sentence-transformers
!pip install -q chromadb
!pip install -q google-generativeai

ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_

In [8]:
import chromadb
import google.generativeai as genai
from sentence_transformers import SentenceTransformer

print("All imports successful")

All imports successful


In [9]:
from google.colab import files
uploaded = files.upload()

Saving Tesla.10k.pdf to Tesla.10k (1).pdf
Saving amazon.10k.pdf to amazon.10k (1).pdf
Saving microsoft.10k.pdf to microsoft.10k (1).pdf
Saving apple.10k.pdf to apple.10k (1).pdf


In [31]:
### configuring Gemini ###
GEMINI_API_KEY = "AQ.Ab8RN6LxL-TwuCv2PHeN76Ntb68e7U-EEjw6Qe_H16J5eD8CpA"
genai.configure(api_key=GEMINI_API_KEY)
LLM = genai.GenerativeModel("gemini-2.5-flash")

In [11]:
!pip install pymupdf

In [12]:
### EXTRACTING TEXT ###
import fitz

documents = []

for file_name in uploaded.keys():

    doc = fitz.open(file_name)

    text = ""

    for page in doc:
        text += page.get_text()

    documents.append({
        "company": file_name.replace(".pdf",""),
        "text": text
    })

    print(f"Finished {file_name}")

print("Loaded:", len(documents))

Finished Tesla.10k (1).pdf
Finished amazon.10k (1).pdf
Finished microsoft.10k (1).pdf
Finished apple.10k (1).pdf
Loaded: 4


In [13]:
### CHUNKING ###
!pip install -q langchain-text-splitters

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = []
for doc in documents:
  split_docs = splitter.split_text(doc["text"])

  for chunk in split_docs:


    chunks.append({
        "company":doc["company"],
        "text": chunk
    })
print("Total Chunks:",len(chunks))

Total Chunks: 1979


In [15]:
### LOADING BGE MODEL AND GENERATING EMBEDDINGS ###
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")


texts = [chunk["text"] for chunk in chunks]

embedding = embedding_model.encode(
    texts,
    batch_size = 32,
    show_progress_bar = True,
    convert_to_numpy= True
)


print("Embedding shape:",embedding.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/62 [00:00<?, ?it/s]

Embedding shape: (1979, 384)


In [16]:
### CREATAING CHROMADB AND STOREING EMBEDDINGS ###
import chromadb
vectordb = chromadb.Client()
collection = vectordb.get_or_create_collection(name = "financial_reports")


documents_text = [chunk["text"] for chunk in chunks]

metadatas = [{
    "company": chunk["company"]}
    for chunk in chunks
]


ids = [
    str(i)
    for i in range(len(chunks))
]


collection.add(
    ids = ids,
    documents = documents_text,
    embeddings = embedding.tolist(),
    metadatas = metadatas
)
print("Stored:", len(documents_text))

Stored: 1979


In [32]:
### BUILDING RETRIEVA FUNCTION ###
def retrieve(query , top_k=5):
  query_embedding = embedding_model.encode(query)

  results = collection.query(query_embeddings=[query_embedding.tolist()],n_results = top_k)

  retrieved = []

  for i in range(len(results["documents"][0])):
    retrieved.append({"text":results["documents"][0][i],
                      "Company":results["metadatas"][0][i]["company"]})
  return retrieved

In [33]:
### TESTING RETRIEVAL FUNCTION ###
docs = retrieve(
    "What are Tesla's major risks?"
)

print(docs[0]["Company"])
print(docs[0]["text"][:500])

Tesla.10k (1)
Though we continue to see increased interest and adoption of electric vehicles, if the market for electric vehicles in general and Tesla vehicles in
particular does not develop as we expect, develops more slowly than we expect, or if demand for our vehicles decreases in our markets or our vehicles
compete with each other, our business, prospects, financial condition and operating results may be harmed.
In addition, electric vehicles still constitute a small percentage of overall vehicle sales. A


In [29]:
### FINAL RAG PIPELINE INCLUDING GEMINI AND PROMPT ###
def ask_financial_assistant(question):

  docs = retrieve(question,top_k = 5)
  context = "\n\n".join(doc["text"] for doc in docs)
  sources = list(set(doc["Company"] for doc in docs))


  prompt = f"""
  You are a senior financial analyst.
  Answer ONLY using provided context.
  if the answer is not found in the context,
  state that clearly.

  CONTEXT:{context}
  QUESTION:{question}

  Provide:
  1. Detailed Answer
  2. Key Insights
  3. Sources Used
  """
  response = LLM.generate_content(prompt)

  return{"answer": response.text,
         "sources": sources}

In [34]:
result = ask_financial_assistant(
    "What are Tesla's major operational risks?"
)

print(result["answer"])
print(result["sources"])

**1. Detailed Answer**

Tesla faces several major operational risks, primarily related to scaling its infrastructure, managing its product lifecycle, and maintaining its workforce dependencies. These include:

*   **Servicing Capacity and Efficiency:** The company risks overburdening its servicing capabilities and parts inventory if it experiences delays in adding servicing capacity or servicing vehicles efficiently, particularly for high-volume models like Model 3 and Model Y. Unforeseen issues with vehicle reliability could exacerbate this.
*   **Supercharger Network Expansion:** There is an ongoing requirement to rapidly increase the number of Supercharger stations and connectors globally to keep pace with the increasing number of Tesla vehicles. Failure to do so could impact customer experience and sales.
*   **Failure to Meet Operational Targets:** Tesla has no assurance that it will be able to ramp its business effectively to meet its sales, delivery, installation, servicing, and

In [36]:
result = ask_financial_assistant(
    "Compare Amazon and Microsoft's growth strategies.")
print(result["answer"])

**1. Detailed Answer**

Based on the provided context:

*   **Amazon's Growth Strategy:** The context does not explicitly outline a singular "growth strategy" for Amazon.com. However, it indicates that Amazon's business involves significant investments in "new business opportunities" and the timing of those investments. Other factors influencing Amazon's operations and implicitly related to growth include the mix of products and services sold to customers, the mix of net sales derived from products versus services, international growth and expansion, and engagement in proposed and completed acquisitions and strategic transactions. These elements suggest a strategy involving diversified investments, managing product/service offerings, global reach, and strategic M&A.

*   **Microsoft's Growth Strategy:** Microsoft's growth strategy is detailed through specific revenue drivers and investments across its segments. Key drivers include:
    *   **Intelligent Cloud:** Revenue growth primaril